### Bronze → Silver: GTFS static schedule dimensions

Typed **current-snapshot** dimensions for `routes`, `trips`, `calendar`, `calendar_dates`,
`stop_times`, `transfers` — to enrich trip updates (route names, service calendars, scheduled times,
and the cancellation baseline for gold).

- **Not SCD2.** Stops is the only dimension we historise; these are the *latest export* only — read
  bronze, filter to `max(export_date)`, cast types, dedup on key, **overwrite**. Idempotent, weekly.
- **Batch, not streaming** — a full refresh of small dims each run (no checkpoint). `stop_times` is the
  heavy one (millions of rows), still a plain overwrite of one export's worth.

> **DRAFT — written against the GTFS data dictionary, not yet validated against real bronze schemas.**
> The helper is defensive: it only casts / dedups on columns that actually exist, so a feed that omits a
> dictionary column won't break it. Depends on the static bronze fan-out having populated the bronze
> tables first. Run as a serverless **job**.

In [ ]:
from pyspark.sql import functions as F

# Catalog is passed as a job parameter (default -> ${var.catalog} per target); the default here
# keeps interactive runs working. Schemas/volumes are the same across dev and prod.
dbutils.widgets.text("catalog", "transport_vic_dev")
CATALOG = dbutils.widgets.get("catalog")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.`02_silver`")


def build_dim(name, key, casts):
    """Latest-export snapshot of a GTFS static file -> typed, deduped, overwritten to silver.
    Defensive: skips casts / key columns that aren't actually present in this feed's bronze."""
    bronze = spark.table(f"{CATALOG}.`01_bronze`.{name}")
    cols   = set(bronze.columns)

    # export_date is a yyyyMMdd string -> lexical max == latest export. Keep only that snapshot.
    latest = bronze.agg(F.max("export_date")).first()[0]
    snap   = bronze.filter(F.col("export_date") == F.lit(latest))

    for c, typ in casts.items():
        if c in cols:
            snap = snap.withColumn(c, F.to_date(c, "yyyyMMdd") if typ == "date" else F.col(c).cast(typ))

    dedup_key = [k for k in key if k in cols]                 # (feed_id, <natural key>) that exist
    snap = (snap.dropDuplicates(dedup_key)
                .withColumn("_silver_ts", F.current_timestamp()))

    (snap.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
         .saveAsTable(f"{CATALOG}.`02_silver`.{name}"))
    print(f"{name}: {snap.count()} rows  (export_date={latest}, key={dedup_key})")

In [ ]:
# Per-file typing: key = (feed_id, natural key); casts = {column: type}. Everything else stays string.
# stop_times.arrival_time/departure_time deliberately stay STRING (GTFS allows values > 24:00:00).
DIMS = [
    ("routes", ["feed_id", "route_id"],
        {"route_type": "int"}),
    ("trips", ["feed_id", "trip_id"],
        {"direction_id": "int", "wheelchair_accessible": "int", "bikes_allowed": "int"}),
    ("calendar", ["feed_id", "service_id"],
        {"monday": "int", "tuesday": "int", "wednesday": "int", "thursday": "int",
         "friday": "int", "saturday": "int", "sunday": "int",
         "start_date": "date", "end_date": "date"}),
    ("calendar_dates", ["feed_id", "service_id", "date"],
        {"date": "date", "exception_type": "int"}),
    ("stop_times", ["feed_id", "trip_id", "stop_sequence"],
        {"stop_sequence": "int", "pickup_type": "int", "drop_off_type": "int",
         "shape_dist_traveled": "double"}),
    ("transfers", ["feed_id", "from_stop_id", "to_stop_id",
                   "from_route_id", "to_route_id", "from_trip_id", "to_trip_id"],
        {"transfer_type": "int", "min_transfer_time": "int"}),
]

for name, key, casts in DIMS:
    build_dim(name, key, casts)

#### Notes
- **Current-snapshot trade-off:** silver holds only the latest export. If the schedule *changes* mid
  capture, a trip-updates row from an earlier day joins the newer schedule. Fine while V/Line's schedule
  is stable (it rarely changes); if it matters later, add an `export_date`-aware point-in-time join
  (bronze already retains every export).
- **Keys:** all `(feed_id, natural key)` — the GTFS id repeats across mode feeds. `transfers` has no clean
  single key, so it dedups on the full from/to set (route/trip qualifiers included when present).
- **`stop_times`** is the large one (one row per stop per trip). Overwrite is still fine weekly; if it
  grows painful, switch to an `export_date`-partitioned append.
- **Pending validation (needs real bronze):** confirm actual column names per file (esp. `transfers`,
  which V/Line may ship with only `from_stop_id`/`to_stop_id`/`transfer_type`/`min_transfer_time`), and
  that `route_type` is a plain int (V/Line may use extended 100-series types — the `int` cast handles both).
- **Bonus:** `routes.route_type` may finally answer the feed-10 "Interstate" mode question for `dim_mode`.